# 260423 사용자 정의 도구 (Custom Tool)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w7_tool_calling/llm_260423_custom_tools.ipynb)

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 1. Tool 에러 처리 - Fallback 패턴

툴은 LLM이 호출하는 외부 의존이라 실패 케이스가 많음:
- API 키 만료, 요청 과다로 호출 차단
- Tavily/DuckDuckGo 같은 외부 서비스 다운 → 응답 지연
- 검색 매칭 실패로 빈 결과

**강의 메모**: `ToolWithFallback` 래퍼 클래스를 만들어, primary 툴이 실패하면 fallback 툴(예: SearchTool 실패 시 DuckDuckGo)로 자동 전환.
- `max_retries`만큼 primary 재시도 → 실패 시 fallback 실행 → 둘 다 실패면 에러 반환
- 응용: 결과가 너무 짧으면 query expansion으로 확장 재검색하는 fallback도 가능

## 2. MCP (Model Context Protocol)

**비유**: 툴이 M개, 사용자/서비스가 N개면 M×N 커넥터가 필요해 비효율적.  
MCP는 표준 프로토콜로 가운데 두어 **M+N**개 연결만 있으면 되도록 만든 규약.

**구조**: `서버`(툴 정의/제공) ↔ `클라이언트`(LLM 쪽에서 툴 호출)

**핵심 포인트**:
- `FastMCP`: MCP 서버를 빠르게 만드는 고수준 라이브러리 (FastAPI 비슷한 느낌)
- `@mcp.tool` 데코레이터로 함수를 MCP 툴로 등록 (LangChain `@tool`과 동일한 컨셉)
- 내부적으로 입출력은 **JSON** 형태로 표준화되어 왕복
- 클라이언트는 `async/await` 비동기로 작성 → 툴 호출이 오래 걸려도 다른 작업 병행 가능
- 첫 연결 시 `initialize` 핸드셰이크로 버전/capabilities 교환
- `mcp_tool_to_langchain` 같은 어댑터로 MCP 툴 → LangChain `StructuredTool` 변환 후 LLM에 바인딩

**docstring 중요**: LLM은 tool description을 보고 어떤 툴을 어떤 인자로 호출할지 결정 → 자세히 작성.

In [ ]:
# ToolWithFallback 예시 - primary 툴 실패 시 fallback 툴로 전환
class ToolWithFallback:
    def __init__(self, primary_tool, fallback_tool, max_retries=2):
        self.primary_tool = primary_tool
        self.fallback_tool = fallback_tool
        self.max_retries = max_retries

    def invoke(self, query):
        # primary 툴을 max_retries 만큼 시도
        for i in range(self.max_retries):
            try:
                # SearchTool은 dict({"query": ...}) 입력, DuckDuckGo는 string 입력
                arg = {"query": query} if isinstance(query, str) else query
                result = self.primary_tool.invoke(arg)
                if result and result.strip():
                    return result
            except Exception as e:
                print(f"[primary] {i+1}회차 실패: {e}")

        # primary 다 실패 → fallback 실행
        try:
            return self.fallback_tool.invoke(query)
        except Exception as e:
            return f"모든 툴 실패: {e}"

# 사용 예: 검색툴이 죽으면 DuckDuckGo로 fallback
# robust = ToolWithFallback(primary_tool=search_tool, fallback_tool=ddg_tool, max_retries=2)
# robust.invoke("모두의연구소")

## 3. 외부 API를 Custom Tool로 만들기

**비유**: API는 식당 주문과 같음. 클라이언트(나)가 **요청(request)** 보내고 서버가 **응답(response)** 줌.

### HTTP 기본
- **Method**: `GET`(조회), `POST`(생성/제출), `PUT`(수정), `DELETE`(삭제)
- **URL**: 어디로 보낼지
- **Header**: 누구인지 - 인증 정보, User-Agent (식당에서 정체불명이면 입장 거부 → 보통 `Mozilla/...` 같은 정상 이름 사용)
- **Body**: 무엇을 보낼지 (POST 시 JSON 등)

### 응답 status code
- `2xx` 성공 (200 OK, 201 Created)
- `4xx` 클라이언트 잘못 (400, 401 인증, 403 권한, 404 없음)
- `5xx` 서버 잘못 (500 Internal Server Error)

### `requests` vs `requests.Session`
- `requests.get/post`: 매 호출마다 핸드셰이크 반복 → 느림
- `Session()`: 최초 1회만 핸드셰이크, 이후 재사용 → **빠름**, 헤더도 한 번 설정하면 유지

### Custom Tool 패턴
1. `requests`로 외부 API 호출하는 함수 작성 (`fetch_user(user_id)` 등)
2. status 체크 → 실패 시 에러 메시지 반환
3. `@tool` 데코레이터 또는 `StructuredTool.from_function`으로 래핑
4. `llm.bind_tools([...])` → LLM이 자동으로 툴 선택 + 인자 채워서 호출
5. tool_calls 반복하며 invoke → ToolMessage로 결과 추가 → 최종 답변 생성

**강의 메모**: LLM 연동 로직은 어제/그제 했던 것과 완전히 동일. 차이는 **툴 내부가 외부 API 호출이라는 점**뿐. JSONPlaceholder 같은 fake API로 프로토타이핑 연습 가능.